In [ ]:
# Cell 1: Setup and Imports
!pip install geopandas rasterio scikit-image shapely tensorflow keras scikit-learn
!pip install networkx xgboost shap folium # Install specialized libraries

import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from skimage.graph import route_through_array

# Deep Learning & ML
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, LSTM, Dropout
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.mixture import GaussianMixture
import shap

# Graph & Geospatial
import networkx as nx
import folium
from shapely.geometry import Point

print("Setup complete. Libraries imported successfully.")

Setup complete. Libraries imported successfully.


In [ ]:
# Cell 2: Synthetic Data Generation & Preprocessing (A. Tasks 1–10)

# --- CONFIGURATION (UPDATED FILE NAME) ---
INDIA_SHAPEFILE = 'gadm41_IND_0.shp'

# 1. Generate Synthetic GPS Data (Task 1)
np.random.seed(42)
N_POINTS = 5000
START_LAT, END_LAT = 15.0, 30.0
START_LON, END_LON = 70.0, 85.0

lat = np.cumsum(np.random.normal(0, 0.01, N_POINTS)) + START_LAT
lon = np.cumsum(np.random.normal(0, 0.01, N_POINTS)) + START_LON

df_gps = pd.DataFrame({
    'lat': lat,
    'lon': lon,
    'timestamp': pd.to_datetime(pd.date_range('2024-01-01', periods=N_POINTS, freq='H')),
    'species_id': np.random.choice(['Tiger', 'Elephant'], N_POINTS),
    'vegetation_index': np.clip(np.random.normal(0.6, 0.2, N_POINTS), 0.1, 1.0),
    'water_proximity': np.clip(np.random.normal(0.3, 0.3, N_POINTS), 0.0, 1.0),
    'habitat_type': np.random.choice(['Forest', 'Water', 'Urban'], N_POINTS, p=[0.6, 0.3, 0.1]),
})
df_gps['presence'] = ((df_gps['vegetation_index'] > 0.5) & (df_gps['water_proximity'] < 0.5)).astype(int)

# 2. Preprocessing (Task 3, 7, 8)
df_gps = df_gps.dropna()
df_gps['habitat_encoded'] = pd.factorize(df_gps['habitat_type'])[0]
features = ['lat', 'lon', 'vegetation_index', 'water_proximity', 'habitat_encoded']
X = df_gps[features].copy()
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)
print(f"Synthetic Data Generated. PCA reduced features from {X_scaled.shape[1]} to {X_pca.shape[1]}.")

Synthetic Data Generated. PCA reduced features from 5 to 3.


/tmp/ipython-input-1210399469.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'timestamp': pd.to_datetime(pd.date_range('2024-01-01', periods=N_POINTS, freq='H')),


In [ ]:
# Cell 3: Habitat Classification (B. Tasks 11–20)

IMAGE_SIZE = 64
NUM_CLASSES = 3
input_shape = (IMAGE_SIZE, IMAGE_SIZE, 3)

# --- Synthetic Image Data Placeholder ---
N_SAMPLES = 100
X_train_img = np.random.rand(N_SAMPLES, IMAGE_SIZE, IMAGE_SIZE, 3)
y_train_img = tf.keras.utils.to_categorical(np.random.randint(0, NUM_CLASSES, N_SAMPLES), num_classes=NUM_CLASSES)

# 1. Transfer learning using pretrained ResNet (Task 16)
base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)

model_resnet = Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    Dense(NUM_CLASSES, activation='softmax')
])

base_model.trainable = False
model_resnet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("ResNet Transfer Learning setup defined. Ready for conceptual training (Task 16).")

ResNet Transfer Learning setup defined. Ready for conceptual training (Task 16).


In [ ]:
# Cell 4: Species Distribution Modeling (C. Tasks 21–30)

# Prepare data
X_ml = pd.DataFrame(X_pca, columns=[f'PC{i}' for i in range(1, 4)])
y_presence = df_gps['presence'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X_ml, y_presence, test_size=0.3, random_state=42)

# 1. Train XGBoost classifier (Task 23)
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)

# 2. Evaluate models with ROC curve (Task 24)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
roc_score = roc_auc_score(y_test, y_pred_proba)
print(f"XGBoost ROC-AUC: {roc_score:.3f}")

# 3. Use SHAP to interpret feature importance (Task 25)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test.iloc[:50, :])
print("SHAP values calculated for feature interpretation.")

# 4. Apply Gaussian Mixture Models (GMM) for clustering habitat zones (Task 27)
gmm = GaussianMixture(n_components=3, random_state=42)
gmm.fit(X_pca)
habitat_clusters = gmm.predict(X_pca)
print("GMM clustering for habitat zones completed (Task 27).")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:27:04] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost ROC-AUC: 0.850
SHAP values calculated for feature interpretation.
GMM clustering for habitat zones completed (Task 27).


In [ ]:
# Cell 5: Synthetic Resistance Raster & LCP (D. Tasks 31–35, 38)

# 1. Define 10 Habitat Nodes (National Parks) - Based on approximate real-world locations
# The coordinates are scaled to fit within the 200x200 RASTER_SIZE for the LCP calculation.
# RASTER_SIZE is 200x200, representing Lat 15-30 and Lon 70-85.
# Scaling: Lat 15->200 (North), Lon 70->0 (West)
RASTER_SIZE = 200
ROWS, COLS = RASTER_SIZE, RASTER_SIZE

# National Park Lat/Lon and their scaled RASTER_INDEX (Row, Col)
# NP_Index = ( Row_Scaled_from_Lat, Col_Scaled_from_Lon )
# Note: Low Lat (South) maps to High Row (Bottom of Raster)
NP_Locations = {
    'NP_1_Hemis': (15, 120),    # Ladakh (High Lat) -> Low Row Index
    'NP_2_Corbett': (50, 150),  # Uttarakhand
    'NP_3_Ranthambore': (80, 80), # Rajasthan
    'NP_4_Gir': (150, 30),      # Gujarat (Low Lon) -> Low Col Index
    'NP_5_Kanha': (140, 170),   # MP (High Lon) -> High Col Index
    'NP_6_Kaziranga': (50, 20), # Assam (East) - NOTE: Simulated location for LCP grid
    'NP_7_Sundarbans': (140, 190),# West Bengal
    'NP_8_Periyar': (190, 100), # Kerala (Low Lat) -> High Row Index
    'NP_9_Bandipur': (180, 85), # Karnataka
    'NP_10_GreatHimalayan': (20, 120)
}
NP_Nodes = list(NP_Locations.keys())
NP_Indices = list(NP_Locations.values())

# 2. Generate Synthetic Resistance Raster (Shared for all paths)
np.random.seed(42)
resistance_array = np.random.rand(ROWS, COLS) * 3

# Add major HIGH-COST BARRIERS (Simulated cities/highways)
resistance_array[90:110, :] = 10     # East-West Highway/Urban Corridor
resistance_array[:, 100:110] = 8.5   # North-South Barrier
resistance_array = np.clip(resistance_array, 1, 10)

all_lcp_coords_indices = [] # Stores indices for all 45 paths
all_lcp_costs = {}          # Stores costs for Cell 6 (Graph Metrics)
path_counter = 0

# 3. Apply Dijkstra's/LCP for all 45 unique pairs (Task 33)
for i in range(len(NP_Indices)):
    for j in range(i + 1, len(NP_Indices)):
        start_point = NP_Indices[i]
        end_point = NP_Indices[j]
        start_name = NP_Nodes[i]
        end_name = NP_Nodes[j]

        path_indices, total_cost = route_through_array(resistance_array,
                                                        start_point,
                                                        end_point,
                                                        fully_connected=True)

        path_indices = np.array(path_indices)
        all_lcp_coords_indices.append(path_indices)
        all_lcp_costs[(start_name, end_name)] = total_cost
        path_counter += 1
        print(f"LCP Calculated for {start_name} to {end_name}. Cost: {total_cost:.2f}")

print(f"\nTotal Corridors Calculated: {path_counter}.")

LCP Calculated for NP_1_Hemis to NP_2_Corbett. Cost: 57.37
LCP Calculated for NP_1_Hemis to NP_3_Ranthambore. Cost: 174.23
LCP Calculated for NP_1_Hemis to NP_4_Gir. Cost: 402.15
LCP Calculated for NP_1_Hemis to NP_5_Kanha. Cost: 345.90
LCP Calculated for NP_1_Hemis to NP_6_Kaziranga. Cost: 205.69
LCP Calculated for NP_1_Hemis to NP_7_Sundarbans. Cost: 358.58
LCP Calculated for NP_1_Hemis to NP_8_Periyar. Cost: 411.39
LCP Calculated for NP_1_Hemis to NP_9_Bandipur. Cost: 393.81
LCP Calculated for NP_1_Hemis to NP_10_GreatHimalayan. Cost: 6.79
LCP Calculated for NP_2_Corbett to NP_3_Ranthambore. Cost: 173.59
LCP Calculated for NP_2_Corbett to NP_4_Gir. Cost: 377.93
LCP Calculated for NP_2_Corbett to NP_5_Kanha. Cost: 294.39
LCP Calculated for NP_2_Corbett to NP_6_Kaziranga. Cost: 239.24
LCP Calculated for NP_2_Corbett to NP_7_Sundarbans. Cost: 304.64
LCP Calculated for NP_2_Corbett to NP_8_Periyar. Cost: 387.17
LCP Calculated for NP_2_Corbett to NP_9_Bandipur. Cost: 369.58
LCP Calculate

In [ ]:
# Cell 6: Graph Metrics and Export (D. Tasks 36, 37, 39, 40)

# 1. Build a Habitat Connectivity Graph (Task 31)
G = nx.Graph()
G.add_weighted_edges_from([
    ('PA_1', 'PA_2', 5), ('PA_2', 'PA_3', 2),
    ('PA_1', 'PA_4', 8), ('PA_3', 'PA_5', 1),
    ('PA_4', 'PA_5', 4), ('PA_2', 'PA_5', 3)])

# 2. Use PageRank to rank habitat importance (Task 36)
pagerank_scores = nx.pagerank(G, weight='weight')
print("\nHabitat Importance (PageRank):")
print({k: f"{v:.4f}" for k, v in pagerank_scores.items()})

# 3. Apply community detection (Task 37)
communities = list(nx.community.label_propagation_communities(G))
print("\nHabitat Communities Detected:")
print(communities)

# 4. Export Connectivity Matrix (Task 40)
connectivity_matrix = nx.to_numpy_array(G)


Habitat Importance (PageRank):
{'PA_1': '0.2636', 'PA_2': '0.2233', 'PA_3': '0.0872', 'PA_4': '0.2448', 'PA_5': '0.1810'}

Habitat Communities Detected:
[{'PA_4', 'PA_3', 'PA_2', 'PA_1', 'PA_5'}]


In [ ]:
# Cell 7: Predictive Movement Modeling (E. Tasks 41–50)

# Helper function for time series data preparation (Task 41)
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length, :2])
    return np.array(X), np.array(y)

SEQ_LENGTH = 10

# Define LSTM Model (Task 41)
model_lstm = Sequential([
    LSTM(32, activation='relu', input_shape=(SEQ_LENGTH, X_scaled.shape[1])),
    Dropout(0.2),
    Dense(2)
])

model_lstm.compile(optimizer='adam', loss='mse')
print("LSTM Model defined for species movement prediction (Task 41).")

LSTM Model defined for species movement prediction (Task 41).


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# Cell 8: Reinforcement Learning (F. Tasks 51–60)

# Environment parameters based on the synthetic resistance_array from Cell 5
ROWS, COLS = resistance_array.shape
NUM_STATES = ROWS * COLS
NUM_ACTIONS = 4
GAMMA = 0.9
ALPHA = 0.1

# Initialize Q-table (Task 52)
Q_table = np.zeros((NUM_STATES, NUM_ACTIONS))

# Function to get reward (Task 56: Introduce human disturbance penalty)
def get_reward(r, c):
    cost = resistance_array[r, c]
    if cost >= 9.0:
        return -100
    return -cost

print("Q-Learning environment structure defined (Task 52).")

Q-Learning environment structure defined (Task 52).


In [ ]:
# Cell 9: Final Visualization (G. Tasks 61–70)

# --- CONFIGURATION ---
INDIA_SHAPEFILE = 'gadm41_IND_0.shp'

# Define geographical bounds used for scaling (MUST match Cell 5 setup)
lat_min, lat_max = 15.0, 30.0
lon_min, lon_max = 70.0, 85.0
ROWS, COLS = 200, 200 # RASTER_SIZE

# Function to scale pixel indices to geographic coordinates
def scale_path_to_coords(path_indices, rows, cols, lat_range, lon_range):
    coords = []
    lat_scale = (lat_range[1] - lat_range[0]) / rows
    lon_scale = (lon_range[1] - lon_range[0]) / cols

    for r, c in path_indices:
        # Note: Latitude mapping is inverted since Row 0 is the top (North/Max Lat)
        lat = lat_range[1] - r * lat_scale
        lon = lon_range[0] + c * lon_scale
        coords.append((lat, lon))
    return coords

# 1. Load the India Map GeoJSON/Shapefile
try:
    india_map = gpd.read_file(INDIA_SHAPEFILE)
except Exception as e:
    print(f"Error loading India shapefile: {e}.")
    india_map = gpd.GeoDataFrame(geometry=[Point(78.0, 20.0)], crs="EPSG:4326")

# 2. Create the Folium Interactive Map
map_center = [22.0, 78.0]
m = folium.Map(location=map_center, zoom_start=5, tiles="cartodbpositron")

# Add the administrative boundaries
folium.GeoJson(
    india_map.to_json(),
    name='India States',
    style_function=lambda x: {'fillColor': 'lightgray', 'color': 'black', 'weight': 1, 'fillOpacity': 0.3}
).add_to(m)

# 3. Overlay ALL 45 Least-Cost Paths (Task 67)
colors = ['red', 'blue', 'green', 'orange', 'purple', 'darkred', 'cadetblue'] # Use varied colors

for i, path_indices in enumerate(all_lcp_coords_indices):
    path_coords = scale_path_to_coords(path_indices, ROWS, COLS, (lat_min, lat_max), (lon_min, lon_max))
    color = colors[i % len(colors)]

    # Plot the corridor line
    folium.PolyLine(
        path_coords,
        color=color,
        weight=2, # Reduced weight for better visualization of many lines
        opacity=0.6,
        tooltip=f'Corridor {i+1}'
    ).add_to(m)

# 4. Add Markers for all 10 Habitat Nodes
for name, (r, c) in NP_Locations.items():
    # Convert NP index (r, c) back to Lat/Lon
    lat_coord, lon_coord = scale_path_to_coords(np.array([(r, c)]), ROWS, COLS, (lat_min, lat_max), (lon_min, lon_max))[0]

    folium.CircleMarker(
        (lat_coord, lon_coord),
        radius=6,
        color='darkgreen',
        fill=True,
        fill_color='lightgreen',
        tooltip=name
    ).add_to(m)


# Display the map and save the report (Tasks 69, 70)
m.save("AI_Wildlife_Corridor_Map_Network.html")
print(f"Interactive Corridor Map updated with {path_counter} paths. Download and view the HTML file.")

Interactive Corridor Map updated with 45 paths. Download and view the HTML file.
